# AI-Assisted ML Activity: Titanic Survival
Upload `train.csv` to Colab (Files panel) first. Path: `/content/train.csv`.

**Part A** = baseline code (Steps 2-3, run as-is, note every issue)
**Part B** = fixed pipeline + 3 algorithms (Step 4)
**Part C** = cross-validation, overfitting check, plots

## PART A: Baseline pipeline (run exactly as given, do NOT fix)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Load
df = pd.read_csv('/content/train.csv')

# 2. EDA
print(df.shape)
df.info()
print(df.describe())
print(df.isnull().sum())
print(df['Survived'].value_counts(normalize=True))

In [ ]:
# 3. Clean and encode
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

X = df.drop('Survived', axis=1)
y = df['Survived']

# 4. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train Logistic Regression
model = LogisticRegression()
model.fit(X_train, y_train)

# 6. Evaluate
pred = model.predict(X_test)
print('Test accuracy:', accuracy_score(y_test, pred))
print('Train accuracy:', model.score(X_train, y_train))
print(classification_report(y_test, pred))

## PART B: Fixed pipeline + three algorithms

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

raw = pd.read_csv('/content/train.csv')

def add_features(d):
    d = d.copy()
    d['Title'] = d['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    d['Title'] = d['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    d['Title'] = d['Title'].where(d['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Rare')
    d['FamilySize'] = d['SibSp'] + d['Parch'] + 1
    d['IsAlone'] = (d['FamilySize'] == 1).astype(int)
    d['HasCabin'] = d['Cabin'].notna().astype(int)
    return d

df2 = add_features(raw)
num_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']
cat_cols = ['Pclass', 'Sex', 'Embarked', 'Title']
bin_cols = ['IsAlone', 'HasCabin']
X = df2[num_cols + cat_cols + bin_cols]
y = df2['Survived']

# stratified split keeps the 38/62 class ratio in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# imputers/scalers live INSIDE the pipeline -> fitted on training folds only (no leakage)
preprocess = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ('bin', 'passthrough', bin_cols),
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, random_state=42),
}
pipes = {n: Pipeline([('prep', preprocess), ('clf', m)]) for n, m in models.items()}

In [ ]:
# Single-split results + overfitting check (train vs test accuracy)
rows = []
for name, pipe in pipes.items():
    pipe.fit(X_train, y_train)
    tr = pipe.score(X_train, y_train)
    te = pipe.score(X_test, y_test)
    rows.append([name, tr, te, tr - te])
    print('=' * 50, name)
    print(classification_report(y_test, pipe.predict(X_test)))
split_df = pd.DataFrame(rows, columns=['Model', 'Train acc', 'Test acc', 'Gap']).round(3)
split_df

## PART C: Cross-validation (use this to pick the best model)

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
cv_rows = []
cv_scores = {}
for name, pipe in pipes.items():
    r = cross_validate(pipe, X, y, cv=cv, scoring='accuracy', return_train_score=True)
    cv_scores[name] = r['test_score']
    cv_rows.append([name, r['train_score'].mean(), r['test_score'].mean(), r['test_score'].std()])
cv_df = pd.DataFrame(cv_rows, columns=['Model', 'CV train acc', 'CV test acc (mean)', 'CV std']).round(3)
cv_df

In [ ]:
# Plots for the report
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].boxplot(cv_scores.values(), labels=cv_scores.keys())
ax[0].set_title('Cross-validated accuracy (5-fold x 3)'); ax[0].set_ylabel('Accuracy')
x = np.arange(len(split_df)); w = 0.35
ax[1].bar(x - w/2, split_df['Train acc'], w, label='Train')
ax[1].bar(x + w/2, split_df['Test acc'], w, label='Test')
ax[1].set_xticks(x); ax[1].set_xticklabels(split_df['Model'], rotation=15)
ax[1].set_ylim(0.6, 1.05); ax[1].set_title('Overfitting check'); ax[1].legend()
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for a, (name, pipe) in zip(axes, pipes.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pipe.predict(X_test))).plot(ax=a, colorbar=False)
    a.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
# OPTIONAL: show the fix for overfitting (limit tree depth) and compare
rf_small = Pipeline([('prep', preprocess),
                     ('clf', RandomForestClassifier(n_estimators=200, max_depth=5,
                                                    min_samples_leaf=3, random_state=42))])
r = cross_validate(rf_small, X, y, cv=cv, scoring='accuracy', return_train_score=True)
print('RF depth=5  train: %.3f  CV test: %.3f' % (r['train_score'].mean(), r['test_score'].mean()))